# ⚡ Spark Multi-Core Utilization Demo

This notebook demonstrates how Apache Spark leverages **multiple CPU cores** to parallelize computation.

| Section | What runs | Expected CPU pattern |
|---------|-----------|----------------------|
| **Part 0** | Pure Python (single thread) | 1 core maxed, rest idle |
| **Part 1** | Spark with `local[1]` (1 worker) | 1 core maxed, rest idle |
| **Part 2** | Spark with `local[*]` (all cores) | **All cores spike simultaneously** |

> 💡 **How to observe spikes:** Open your system monitor / Activity Monitor / `htop` in a terminal and watch per-core CPU graphs while running each cell.

## 🔧 Setup

In [ ]:
import subprocess, sys

# Install dependencies if needed
for pkg in ["pyspark", "psutil", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os, sys, time, math, psutil, multiprocessing
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyspark.sql import SparkSession

# ── Windows fix: Spark calls "python3" by default which does not exist on Windows.
# Point both driver and worker to the exact Python running this notebook.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

CPU_COUNT = multiprocessing.cpu_count()
print(f"✅ Setup complete")
print(f"🖥️  Logical CPU cores available: {CPU_COUNT}")
print(f"📦 PySpark version: {__import__('pyspark').__version__}")

## 🧮 The Task: Count Primes via Trial Division

We count prime numbers in a large range using **trial division** — a pure CPU-bound operation with no I/O.  
Each chunk of numbers is completely independent, making it **embarrassingly parallel**.

The same function runs in all three parts so results are directly comparable.

In [ ]:
# ── Shared CPU-bound kernel ──────────────────────────────────────────────────
def count_primes_in_range(args):
    """Count primes in [start, end) using trial division — pure CPU work."""
    start, end = args
    count = 0
    for n in range(max(2, start), end):
        is_prime = True
        for d in range(2, int(math.isqrt(n)) + 1):
            if n % d == 0:
                is_prime = False
                break
        if is_prime:
            count += 1
    return count

# ── Work configuration ───────────────────────────────────────────────────────
# UPPER_LIMIT is large enough so each partition takes ~5-10 seconds.
# That guarantees all cores are running AT THE SAME TIME — which is what
# produces simultaneous spikes on the CPU monitor.
#
# Rule of thumb: each partition should take ≥ 5 s.
# Adjust UPPER_LIMIT up if tasks finish too fast on your machine.
UPPER_LIMIT    = 3_000_000   # Count primes below this number

# ONE partition per core — each core gets one big, long-running task.
# If NUM_PARTITIONS > CPU_COUNT, tasks finish in waves (not a clean spike).
NUM_PARTITIONS = CPU_COUNT

CHUNK_SIZE = UPPER_LIMIT // NUM_PARTITIONS

# Pre-build (start, end) ranges — one per core
ranges = [
    (i * CHUNK_SIZE, min((i + 1) * CHUNK_SIZE, UPPER_LIMIT))
    for i in range(NUM_PARTITIONS)
]

est_sec_per_core = (UPPER_LIMIT / 1_000_000) * 4   # rough empirical estimate
print(f"Range           : 2 … {UPPER_LIMIT:,}")
print(f"Partitions      : {NUM_PARTITIONS}  (= CPU cores, so tasks run concurrently)")
print(f"Chunk size      : ~{CHUNK_SIZE:,} numbers each")
print(f"Est. time/core  : ~{est_sec_per_core:.0f}s  (tune UPPER_LIMIT if too fast/slow)")


---
## Part 0 — Pure Python (Single Thread)

Plain Python processes chunks **sequentially** in a `for` loop.  
Only **one core** is used — the others sit idle.

```
Core 1  ████████████████████████████████  (100%)
Core 2  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (idle)
Core 3  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (idle)
Core N  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (idle)
```

In [ ]:
print("▶ Part 0: Pure Python — single-threaded loop")
print("  👀 Watch your CPU monitor — only ONE core should spike!\n")

t0 = time.perf_counter()
total_python = sum(count_primes_in_range(r) for r in ranges)
t_python = time.perf_counter() - t0

print(f"  Primes found : {total_python:,}")
print(f"  ⏱  Time      : {t_python:.2f}s")

---
## Part 1 — Spark with 1 Core (`local[1]`)

Spark is initialised with **`local[1]`** — exactly one executor thread.  
Spark overhead is introduced, but parallelism is still absent.  
Result: same single-core spike as pure Python, just slightly slower.

```
Core 1  ████████████████████████████████  (100%)
Core 2  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░  (idle)
        └─ Spark overhead but NO parallelism
```

In [ ]:
# ── Spark: 1 core ────────────────────────────────────────────────────────────
spark1 = (
    SparkSession.builder
    .master("local[1]")                              # ← hard-coded single thread
    .appName("PrimeCount_1Core")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", "2g")
    .config("spark.default.parallelism", "1")        # ← force 1 task at a time
    .getOrCreate()
)
spark1.sparkContext.setLogLevel("ERROR")

print("▶ Part 1: Spark local[1] — single executor thread")
print("  👀 Watch your CPU monitor — still only ONE core should spike!\n")

rdd1     = spark1.sparkContext.parallelize(ranges, NUM_PARTITIONS)
t0       = time.perf_counter()
total1   = rdd1.map(count_primes_in_range).sum()
t_spark1 = time.perf_counter() - t0

spark1.stop()

print(f"  Primes found : {int(total1):,}")
print(f"  ⏱  Time      : {t_spark1:.2f}s  (includes Spark overhead vs {t_python:.2f}s pure Python)")


---
## Part 2 — Spark with ALL Cores (`local[*]`)

Spark is now initialised with **`local[*]`** — one executor thread **per logical CPU core**.  
All partitions are dispatched to worker threads concurrently.

```
Core 1  ████████████████████  (100%)
Core 2  ████████████████████  (100%)
Core 3  ████████████████████  (100%)
Core N  ████████████████████  (100%)
        └─ ALL cores spike simultaneously ✅
```

> The more cores your machine has, the bigger the speedup relative to Part 1.

In [ ]:
# ── Spark: all cores ─────────────────────────────────────────────────────────
spark_all = (
    SparkSession.builder
    .master(f"local[{CPU_COUNT}]")                    # ← one thread per logical core
    .appName("PrimeCount_AllCores")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.driver.memory", "2g")
    .config("spark.default.parallelism", str(CPU_COUNT))  # ← match thread count
    .getOrCreate()
)
spark_all.sparkContext.setLogLevel("ERROR")

print(f"▶ Part 2: Spark local[{CPU_COUNT}] — {CPU_COUNT} executor thread(s)")
print(f"  👀 Watch your CPU monitor — ALL {CPU_COUNT} core(s) should spike NOW!\n")

rdd_all     = spark_all.sparkContext.parallelize(ranges, NUM_PARTITIONS)
t0          = time.perf_counter()
total_all   = rdd_all.map(count_primes_in_range).sum()
t_spark_all = time.perf_counter() - t0

spark_all.stop()

print(f"  Primes found : {int(total_all):,}")
print(f"  ⏱  Time      : {t_spark_all:.2f}s")


---
## 📊 Results & Speedup Analysis

In [ ]:
# ── Correctness check ────────────────────────────────────────────────────────
assert total_python == int(total1) == int(total_all), "❌ Results differ — something went wrong!"
print(f"✅ All three methods agree: {total_python:,} primes below {UPPER_LIMIT:,}\n")

# ── Speedup metrics ──────────────────────────────────────────────────────────
speedup_vs_python  = t_python    / t_spark_all
speedup_vs_spark1  = t_spark1    / t_spark_all
efficiency         = speedup_vs_spark1 / CPU_COUNT * 100

print(f"{'Method':<32} {'Time':>8}  {'vs All-Core Spark':>18}")
print("-" * 62)
print(f"{'Pure Python (1 thread)':<32} {t_python:>7.2f}s  {t_python/t_spark_all:>17.2f}×")
print(f"{'Spark local[1]':<32} {t_spark1:>7.2f}s  {t_spark1/t_spark_all:>17.2f}×")
print(f"{'Spark local[*] (all cores)':<32} {t_spark_all:>7.2f}s  {'1.00':>17}×  ← fastest")
print()
print(f"  Parallelisation speedup  : {speedup_vs_spark1:.2f}× (Spark 1-core → all-core)")
print(f"  CPU cores used           : {CPU_COUNT}")
print(f"  Parallel efficiency      : {efficiency:.1f}%  (100% = perfect linear scaling)")


In [ ]:
# ── Visualisation ────────────────────────────────────────────────────────────
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    f"Spark Multi-Core Demo — Prime Counting (2 … {UPPER_LIMIT:,})",
    fontsize=14, fontweight="bold", y=1.02
)

labels  = ["Pure Python\n(1 thread)", "Spark\nlocal[1]", f"Spark\nlocal[{CPU_COUNT}] ★"]
times   = [t_python, t_spark1, t_spark_all]
colors  = ["#e74c3c", "#e67e22", "#27ae60"]

# --- Bar chart: wall-clock time ---
ax1 = axes[0]
bars = ax1.bar(labels, times, color=colors, edgecolor="white", linewidth=1.5, width=0.5)
for bar, t in zip(bars, times):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(times) * 0.01,
        f"{t:.2f}s",
        ha="center", va="bottom", fontweight="bold", fontsize=11
    )
ax1.set_ylabel("Wall-clock time (seconds)", fontsize=11)
ax1.set_title("Execution Time", fontsize=12)
ax1.set_ylim(0, max(times) * 1.18)
ax1.grid(axis="y", alpha=0.3)
ax1.spines[["top", "right"]].set_visible(False)

# --- Simulated CPU core utilisation heatmap ---
ax2 = axes[1]
n_cores_display = max(CPU_COUNT, 4)

util = [
    [100] + [0] * (n_cores_display - 1),
    [100] + [0] * (n_cores_display - 1),
    [100] * CPU_COUNT + [0] * (n_cores_display - CPU_COUNT)
]

util_arr = np.array(util, dtype=float)
im = ax2.imshow(util_arr, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100, interpolation="nearest")

ax2.set_yticks(range(3))
ax2.set_yticklabels(labels, fontsize=9)
ax2.set_xticks(range(n_cores_display))
ax2.set_xticklabels([f"Core {i+1}" for i in range(n_cores_display)], fontsize=9)
ax2.set_title("Simulated CPU Core Utilisation", fontsize=12)

for row in range(3):
    for col in range(n_cores_display):
        val = util_arr[row, col]
        ax2.text(col, row, f"{int(val)}%",
                 ha="center", va="center",
                 fontsize=9, fontweight="bold",
                 color="white" if val > 50 else "#444")

plt.colorbar(im, ax=ax2, label="CPU Utilisation (%)", shrink=0.8)

speedup_vs_spark1 = t_spark1 / t_spark_all
efficiency = speedup_vs_spark1 / CPU_COUNT * 100
fig.text(
    0.5, -0.04,
    f"★ Spark local[{CPU_COUNT}] is {speedup_vs_spark1:.2f}× faster than local[1]  "
    f"| Parallel efficiency: {efficiency:.1f}%",
    ha="center", fontsize=11, style="italic", color="#27ae60"
)

plt.tight_layout()
plt.savefig("spark_multicore_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Chart saved to spark_multicore_results.png")


---
## 🧵 How to Observe the CPU Spikes Yourself

Run this notebook and simultaneously open a system monitor:

| OS | Tool |
|----|------|
| **Linux** | `htop` in a terminal, or GNOME System Monitor → Resources tab |
| **macOS** | Activity Monitor → CPU tab (choose "Show All Processes") |
| **Windows** | Task Manager → Performance → CPU (right-click graph → "Change graph to → Logical processors") |

### What you should see

| Cell running | Expected CPU graph |
|-------------|--------------------|
| **Part 0 / Part 1** | One core bar jumps to ~100%, all others stay flat |
| **Part 2** | **Every core bar** simultaneously jumps to ~100% for the duration of the job |

---
## 🔑 Key Takeaways

1. **`local[1]`** — Spark runs on a single thread; no parallelism, just overhead compared to pure Python.
2. **`local[*]`** — Spark spawns one thread per logical CPU core; each core handles its own partition concurrently.
3. **Speedup ≈ number of cores** for embarrassingly parallel, CPU-bound workloads.
4. In a real cluster, `local[*]` becomes a distributed `spark://master:7077` URL — the code stays identical, the scale changes.